# EB-ALFRED Results Analysis

This notebook analyzes the experimental results from Qwen3.5-397B-A17B baseline.

**Note**: Invalid episodes (planner_steps == 0) are filtered out from analysis.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Set the results directory
results_dir = Path(
    '../running/eb_alfred/claude-sonnet-4-20250514_visible/base/results')

# Get all episode result files
result_files = sorted(results_dir.glob('episode_*_final_res.json'))
print(f'Found {len(result_files)} result files')

In [ ]:
# Load all results
all_results = []
for file_path in result_files:
    with open(file_path, 'r') as f:
        data = json.load(f)
        episode_num = int(file_path.stem.split('_')[1])
        data['episode'] = episode_num
        all_results.append(data)

# Create a DataFrame with key metrics
df = pd.DataFrame([{
    'episode':
    r['episode'],
    'task_success':
    r.get('task_success', 0),
    'task_progress':
    r.get('task_progress', 0),
    'planner_steps':
    r.get('planner_steps', 0),
    'num_steps':
    r.get('num_steps', 0),
    'num_invalid_actions':
    r.get('num_invalid_actions', 0),
    'num_invalid_action_ratio':
    r.get('num_invalid_action_ratio', 0),
    'eqa_accuracy':
    r.get('eqa_accuracy', 0),
    'eqa_total':
    r.get('eqa_total', 0),
    'eqa_correct':
    r.get('eqa_correct', 0),
    'reward':
    r.get('reward', 0),
    'empty_plan':
    r.get('empty_plan', False),
    'instruction':
    r.get('instruction', False),
    'episode_elapsed_seconds':
    r.get('episode_elapsed_seconds', 0)
} for r in all_results])

df = df.sort_values('episode').reset_index(drop=True)

# Filter out invalid episodes (planner_steps == 0)
df_full = df.copy()
df = df[df['planner_steps'] > 0].reset_index(drop=True)

print(f'=== Data Filtering ===')
print(f'Total episodes (raw): {len(df_full)}')
print(f'Invalid episodes (planner_steps == 0): {(df_full["planner_steps"] == 0).sum()}')
print(f'Valid episodes (planner_steps > 0): {len(df)}')
print(f'Filtered episode numbers: {df_full[df_full["planner_steps"] == 0]["episode"].tolist()}')

# Success episodes analysis
success_episodes = df[df['task_success'] == 1.0]
print(f'\n=== Success Episodes Analysis ({len(success_episodes)} episodes) ===')
if len(success_episodes) > 0:
    print(f'Average planner steps (success): {success_episodes["planner_steps"].mean():.2f}')
    print(f'Average EQA accuracy (success): {success_episodes["eqa_accuracy"].mean():.4f}')
    print(f'Success episode numbers: {sorted(success_episodes["episode"].tolist())}')
else:
    print('No successful episodes')

# Final Summary
print('\n' + '='*60)
print('FINAL SUMMARY - Qwen3.5-397B-A17B Baseline Results')
print('='*60)
print(f'Data filtering: {len(df_full)} total episodes, {len(df)} valid episodes (planner_steps > 0)')
print(f'\nMetrics on valid episodes:')
print(f'Task Success Rate:     {df["task_success"].mean():.2%} ({df["task_success"].sum():.0f}/{len(df)})')
print(f'Average Task Progress: {df["task_progress"].mean():.2%}')
print(f'Total EQA Questions:   {df["eqa_total"].sum():.0f}')
print(f'Total EQA Correct: {df["eqa_correct"].sum()}')
print(f'Average EQA Accuracy:  {df["eqa_accuracy"].mean():.2%}')
print(f'Average Planner Steps: {df["planner_steps"].mean():.1f}')
print(f'Empty Plans:           {df["empty_plan"].sum()} ({df["empty_plan"].sum()/len(df)*100:.1f}%)')
print('='*60)

In [ ]:
# Detailed Statistics
stats_df = pd.DataFrame({
    'Metric': ['Task Success Rate', 'Average Task Progress', 'Average Planner Steps', 
               'Average Environment Steps', 'Average EQA Accuracy', 
               'Average Invalid Actions', 'Average Invalid Action Ratio',
               'Average Reward', 'Average Episode Time (s)', 'Empty Plan Episodes'],
    'Value': [
        f"{df['task_success'].mean():.4f}",
        f"{df['task_progress'].mean():.4f}",
        f"{df['planner_steps'].mean():.2f}",
        f"{df['num_steps'].mean():.2f}",
        f"{df['eqa_accuracy'].mean():.4f}",
        f"{df['num_invalid_actions'].mean():.2f}",
        f"{df['num_invalid_action_ratio'].mean():.4f}",
        f"{df['reward'].mean():.2f}",
        f"{df['episode_elapsed_seconds'].mean():.2f}",
        f"{df['empty_plan'].sum()} ({df['empty_plan'].sum()/len(df)*100:.1f}%)"
    ]
})


In [ ]:
# Per-episode breakdown table
print('\n=== Per-Episode Breakdown (Filtered Data) ===')
display_df = df[[
    'episode', 'task_success', 'task_progress', 'planner_steps',
    'eqa_accuracy', 'eqa_correct', 'eqa_total', 'instruction'
]].copy()
display_df['success_status'] = display_df['task_success'].map({0.0: 'FAIL', 1.0: 'SUCCESS'})
display_df['eqa_accuracy_str'] = display_df.apply(
    lambda x: f"{x['eqa_correct']:.0f}/{x['eqa_total']:.0f} ({x['eqa_accuracy']:.2%})", axis=1)

pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
display_df[[
    'episode', 'success_status', 'task_progress', 'planner_steps',
    'eqa_accuracy_str', 'instruction'
]]
